<a href="https://colab.research.google.com/github/LIBY70/Data-Analysis/blob/main/ida_week6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab6: 탐색적 데이터 분석 (EDA)

본 실습 자료는 『처음 시작하는 파이썬 데이터 분석』 (안기수, 생능출판사)의 내용과 『배워서 바로 써먹는 데이터 분석 with 파이썬』 (설진욱, 생능북스)의 데이터를 활용하여 제작되었습니다.

## 0. 라이브러리 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

### 한글 폰트 설정

In [ ]:
!apt-get install -y fonts-nanum

In [ ]:
import platform

if platform.system() == 'Darwin':       # macOS
    mpl.rc('font', family='AppleGothic')
elif platform.system() == 'Windows':    # Windows
    mpl.rc('font', family='Malgun Gothic')
else:                                   # Linux / Colab
    import matplotlib.font_manager as fm
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    mpl.rc('font', family='NanumGothic')

mpl.rc('axes', unicode_minus=False)     # 마이너스 기호 깨짐 방지
print('폰트 설정 완료:', mpl.rcParams['font.family'])

---
## 1. 탐색적 데이터 분석(EDA) 소개

**탐색적 데이터 분석(Exploratory Data Analysis; EDA)**은 데이터의 주요 특성을 요약하기 위해 **시각화**를 사용하며, 데이터의 구조·패턴·변수 간의 관계를 이해하고 데이터에 존재하는 특이값과 특징을 알아보기 위하여 사용한다.

이번 실습에서는 국내 치킨 프랜차이즈 4개 브랜드(**처가집, 굽네, 페리카나, 네네**)의 전국 매장 데이터를 활용하여 브랜드별·지역별 분포 특성을 탐색한다.

### 탐색 내용

| 탐색 | 주제 |
|---|---|
| 기본 정보 | 데이터 구조, 결측값, 브랜드 및 지역 분포 |
| 탐색 1 | 브랜드별 수도권 vs 비수도권 매장 비율 비교 |
| 탐색 2 | 지역 유형별 브랜드 분포 비교 |
| 탐색 3 | 시도별 매장 수 분포 |
| 탐색 4 | 광역시별 브랜드 분포 비교 |

---
## 2. 데이터 불러오기 및 기본 정보

### 2.1 변수와 값

`chicken_brands.csv`: 국내 치킨 프랜차이즈 4개 브랜드의 전국 매장 정보

| 변수명 | 설명 | 데이터 타입 |
|---|---|---|
| `brand` | 브랜드명 (cheogajip / goobne / pelicana / nene) | 범주형 |
| `store` | 매장명 | 범주형 |
| `province` | 시·도 | 범주형 |
| `district` | 군·구 | 범주형 |
| `address` | 도로명 주소 | 범주형 |
| `phone` | 전화번호 | 범주형 |

In [ ]:
df = pd.read_csv('chicken_brands.csv', encoding='utf-8-sig')
df = df.drop(columns=['Unnamed: 0'])
df.head()

In [ ]:
df.info()

### 2.2 결측값 확인 및 처리

`isnull().sum()`으로 각 변수의 결측값 수를 확인

In [ ]:
df.isnull().sum()

`province` column은 잘못된 값(시·군 이름이 입력된 경우)과 결측값이 존재
- 잘못된 값 → 해당 시·도명으로 수정
- 결측값 → `address`에서 시·도명을 추출하여 보완

In [ ]:
# 잘못된 province 값 수정
province_fix = {'충주시': '충청북도', '양산시': '경상남도', '순천시': '전라남도', '대전광영시': '대전광역시'}
df['province'] = df['province'].replace(province_fix)

# province가 NaN인 경우 address에서 시·도 추출
province_abbr = {
    '서울': '서울특별시', '경기': '경기도', '인천': '인천광역시',
    '부산': '부산광역시', '대구': '대구광역시', '울산': '울산광역시',
    '광주': '광주광역시', '대전': '대전광역시',
    '경상남': '경상남도', '경남': '경상남도',
    '경상북': '경상북도', '경북': '경상북도',
    '전라남': '전라남도', '전남': '전라남도',
    '전라북': '전라북도', '전북': '전라북도',
    '충청남': '충청남도', '충남': '충청남도',
    '충청북': '충청북도', '충북': '충청북도',
    '강원': '강원도', '제주': '제주특별자치도', '세종': '세종특별자치시'
}
for idx in df[df['province'].isna()].index:
    addr = df.loc[idx, 'address']
    for k, v in province_abbr.items():
        if str(addr).startswith(k):
            df.loc[idx, 'province'] = v
            break

print('처리 후 결측값:')
print(df.isnull().sum())

### 2.3 파생 변수 생성

분석에 필요한 파생 변수를 추가한다.

| 파생 변수 | 설명 |
|---|---|
| `브랜드` | brand를 한글 이름으로 변환 |
| `수도권` | province가 서울·경기·인천이면 `수도권`, 나머지는 `비수도권` |
| `지역유형` | `수도권` / `광역시(비수도권)` / `기타 지방` |

In [ ]:
# 브랜드 한글명
brand_kr = {'cheogajip': '처가집', 'goobne': '굽네', 'pelicana': '페리카나', 'nene': '네네'}
df['브랜드'] = df['brand'].map(brand_kr)

# 수도권 여부
capital_region = ['서울특별시', '경기도', '인천광역시']
df['수도권'] = df['province'].isin(capital_region).map({True: '수도권', False: '비수도권'})

# 지역 유형
metro_cities = ['부산광역시', '대구광역시', '울산광역시', '광주광역시', '대전광역시']

def get_region_type(province):
    if province in capital_region:
        return '수도권'
    elif province in metro_cities:
        return '광역시(비수도권)'
    else:
        return '기타 지방'

df['지역유형'] = df['province'].apply(get_region_type)

df[['brand', '브랜드', 'province', '수도권', '지역유형']].head(10)

### 2.4 기본 분포 확인

`value_counts()`로 범주형 변수의 사례 수와 구성비를 확인한다.

In [ ]:
# 브랜드별 매장 수
brand_cnt = df['브랜드'].value_counts()
brand_cnt

In [ ]:
# 브랜드별 매장 구성비(%)
(brand_cnt / len(df) * 100).round(1)

In [ ]:
# 브랜드별 매장 수 파이차트 + 막대그래프
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 파이차트
axes[0].pie(
    brand_cnt,
    labels=brand_cnt.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=sns.color_palette('pastel')
)
axes[0].set_title('브랜드별 매장 구성비')

# 막대그래프
axes[1].bar(brand_cnt.index, brand_cnt.values, color=sns.color_palette('pastel'))
axes[1].set_title('브랜드별 매장 수')
axes[1].set_xlabel('브랜드')
axes[1].set_ylabel('매장 수')
for i, v in enumerate(brand_cnt.values):
    axes[1].text(i, v + 5, str(v), ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# 수도권 여부 사례 수
capital_cnt = df['수도권'].value_counts()
capital_cnt

In [ ]:
# 수도권 구성비(%)
(capital_cnt / len(df) * 100).round(1)

In [ ]:
# 지역유형별 매장 수 파이차트
region_cnt = df['지역유형'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(
    region_cnt,
    labels=region_cnt.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=sns.color_palette('Set2')
)
axes[0].set_title('지역유형별 매장 구성비')

axes[1].barh(region_cnt.index, region_cnt.values, color=sns.color_palette('Set2'))
axes[1].set_title('지역유형별 매장 수')
axes[1].set_xlabel('매장 수')
for i, v in enumerate(region_cnt.values):
    axes[1].text(v + 5, i, str(v), va='center')

plt.tight_layout()
plt.show()

In [ ]:
# 시도별 전체 매장 수 (상위 10개)
province_cnt = df['province'].value_counts().head(10)

plt.figure(figsize=(10, 5))
plt.bar(province_cnt.index, province_cnt.values, color=sns.color_palette('Blues_r', 10))
plt.title('시·도별 전체 매장 수 (상위 10개)')
plt.xlabel('시·도')
plt.ylabel('매장 수')
plt.xticks(rotation=30, ha='right')
for i, v in enumerate(province_cnt.values):
    plt.text(i, v + 5, str(v), ha='center')
plt.tight_layout()
plt.show()

---
## [탐색 1] 브랜드별 수도권 vs 비수도권 매장 비율 비교

브랜드에 따라 수도권과 비수도권 매장의 비율이 다른지 확인한다.

`sns.countplot()`을 사용하면 범주(수도권 여부)에 따라 구분된 막대그래프를 한 번에 그릴 수 있다.

In [ ]:
# 브랜드별 수도권/비수도권 매장 수 비교
brand_order = ['처가집', '굽네', '페리카나', '네네']

plt.figure(figsize=(9, 5))
sns.countplot(data=df, x='브랜드', hue='수도권', order=brand_order, palette='Set1')
plt.title('브랜드별 수도권 vs 비수도권 매장 수')
plt.xlabel('브랜드')
plt.ylabel('매장 수')
plt.legend(title='지역')
plt.tight_layout()
plt.show()

브랜드별로 수도권과 비수도권의 정확한 매장 수를 확인하려면 **교차표(cross table)**가 필요하다.

`pandas.crosstab()`을 이용하면 두 범주형 변수의 빈도표를 계산할 수 있다.

In [ ]:
# 브랜드 × 수도권 교차표
ct1 = pd.crosstab(df['브랜드'], df['수도권'])
ct1

In [ ]:
# 브랜드별 비율 계산
ct1_ratio = ct1.div(ct1.sum(axis=1), axis=0) * 100
ct1_ratio = ct1_ratio.round(1)
ct1_ratio

In [ ]:
# 브랜드별 수도권 비율 막대그래프
capital_ratio = ct1_ratio['수도권'].sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 수도권 비율
axes[0].bar(capital_ratio.index, capital_ratio.values, color=sns.color_palette('Set1', 4))
axes[0].set_title('브랜드별 수도권 매장 비율')
axes[0].set_xlabel('브랜드')
axes[0].set_ylabel('비율 (%)')
axes[0].set_ylim(0, 60)
for i, v in enumerate(capital_ratio.values):
    axes[0].text(i, v + 0.5, f'{v}%', ha='center')

# 비수도권 비율
non_capital_ratio = ct1_ratio['비수도권'].sort_values(ascending=False)
axes[1].bar(non_capital_ratio.index, non_capital_ratio.values, color=sns.color_palette('Set2', 4))
axes[1].set_title('브랜드별 비수도권 매장 비율')
axes[1].set_xlabel('브랜드')
axes[1].set_ylabel('비율 (%)')
axes[1].set_ylim(0, 70)
for i, v in enumerate(non_capital_ratio.values):
    axes[1].text(i, v + 0.5, f'{v}%', ha='center')

plt.tight_layout()
plt.show()

교차표 결과를 정리하면 다음과 같다.

| 브랜드 | 전체 매장 수 | 비수도권 | 수도권 | 수도권 비율 |
|---|---|---|---|---|
| 처가집 | 1,247 | 780 | 467 | 37.5% |
| 굽네 | 1,110 | 611 | 499 | 45.0% |
| 페리카나 | 1,075 | 551 | 524 | 48.7% |
| 네네 | 1,057 | 609 | 448 | 42.4% |

**처가집**은 수도권 비율이 37.5%로 4개 브랜드 중 비수도권 집중도가 가장 높고, **페리카나**는 48.7%로 수도권과 비수도권 매장 비율이 가장 균형적이다.

---
## [탐색 2] 지역 유형별 브랜드 분포 비교

수도권·광역시(비수도권)·기타 지방의 세 가지 지역 유형에서 브랜드 분포가 어떻게 다른지 확인한다.

In [ ]:
# 지역유형별 브랜드 매장 수 비교
region_order = ['수도권', '광역시(비수도권)', '기타 지방']

plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='지역유형', hue='브랜드', order=region_order,
              hue_order=brand_order, palette='tab10')
plt.title('지역 유형별 브랜드 매장 수')
plt.xlabel('지역 유형')
plt.ylabel('매장 수')
plt.legend(title='브랜드')
plt.tight_layout()
plt.show()

`sns.catplot()`은 `countplot()`과 유사하지만, 지역유형과 브랜드를 각각 x축과 열(col)로 배치하여 **브랜드별 지역 분포를 비교하는 서브플롯**을 간결하게 생성할 수 있다.

In [ ]:
# 브랜드별 지역유형 분포 서브플롯 (catplot)
g = sns.catplot(
    data=df,
    x='지역유형',
    col='브랜드', col_order=brand_order,
    kind='count', order=region_order,
    palette='Set2', height=4, aspect=0.8
)
g.set_axis_labels('지역 유형', '매장 수')
g.set_titles(col_template='{col_name}')
g.set_xticklabels(rotation=20)
g.figure.suptitle('브랜드별 지역 유형 매장 수', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# 브랜드 × 지역유형 교차표
ct2 = pd.crosstab(df['브랜드'], df['지역유형'])[region_order]
ct2

In [ ]:
# 브랜드별 지역유형 비율
ct2_ratio = ct2.div(ct2.sum(axis=1), axis=0) * 100
ct2_ratio.round(1)

In [ ]:
# 브랜드별 지역유형 비율 누적 막대그래프
ct2_ratio_sorted = ct2_ratio.loc[brand_order]

fig, ax = plt.subplots(figsize=(9, 5))
colors = sns.color_palette('Set2', 3)

bottom = np.zeros(4)
for i, col in enumerate(region_order):
    ax.bar(ct2_ratio_sorted.index, ct2_ratio_sorted[col],
           bottom=bottom, label=col, color=colors[i])
    bottom += ct2_ratio_sorted[col].values

ax.set_title('브랜드별 지역 유형 비율')
ax.set_xlabel('브랜드')
ax.set_ylabel('비율 (%)')
ax.legend(title='지역유형', loc='upper right')
plt.tight_layout()
plt.show()

교차표 비율을 정리하면 다음과 같다.

| 브랜드 | 수도권 | 광역시(비수도권) | 기타 지방 |
|---|---|---|---|
| 처가집 | 37.5% | 21.1% | 41.5% |
| 굽네 | 45.0% | 20.8% | 34.2% |
| 페리카나 | 48.7% | 12.7% | 38.5% |
| 네네 | 42.4% | 18.1% | 39.5% |

- **처가집**은 기타 지방 비율(41.5%)이 4개 브랜드 중 가장 높아 지방 중소도시에 집중하는 전략을 취하고 있다.
- **굽네**는 수도권 비율(45.0%)이 두 번째로 높으면서 기타 지방 비율(34.2%)이 가장 낮다.
- **페리카나**는 수도권 비율(48.7%)이 가장 높고 광역시(비수도권) 비율(12.7%)이 가장 낮다.

---
## [탐색 3] 시도별 매장 수 분포 비교

시·도별 매장 수의 분포와 브랜드별 시도 분포가 어떻게 다른지 확인한다.

In [ ]:
# 시도별 전체 매장 수
province_cnt_all = df['province'].value_counts()

plt.figure(figsize=(12, 5))
plt.bar(province_cnt_all.index, province_cnt_all.values, color=sns.color_palette('Blues_r', len(province_cnt_all)))
plt.title('시·도별 전체 매장 수')
plt.xlabel('시·도')
plt.ylabel('매장 수')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

`sns.FacetGrid()`는 행 또는 열 방향으로 서로 다른 조건(브랜드)별 서브플롯을 작성할 때 유용하다.
각 서브플롯에 그릴 그래프는 `.map()` 메서드로 지정한다.

In [ ]:
# 상위 10개 시도 목록 추출
top10_province = df['province'].value_counts().head(10).index.tolist()
df_top10 = df[df['province'].isin(top10_province)].copy()
df_top10['province'] = pd.Categorical(df_top10['province'], categories=top10_province, ordered=True)

# 브랜드별 시도 매장 수 분포 (FacetGrid)
g = sns.FacetGrid(df_top10, col='브랜드', col_order=brand_order, col_wrap=2, height=4, aspect=1.4)
g.map_dataframe(
    lambda data, **kw: sns.countplot(data=data, x='province', order=top10_province,
                                     palette='Blues_r', **kw)
)
g.set_axis_labels('시·도', '매장 수')
g.set_titles(col_template='{col_name}')
for ax in g.axes.flat:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right')
g.figure.suptitle('브랜드별 상위 10개 시도 매장 수', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 상위 5개 시도에서의 브랜드 비교
top5_province = df['province'].value_counts().head(5).index.tolist()
df_top5 = df[df['province'].isin(top5_province)]

plt.figure(figsize=(11, 5))
sns.countplot(data=df_top5, x='province', hue='브랜드',
              order=top5_province, hue_order=brand_order, palette='tab10')
plt.title('상위 5개 시도에서의 브랜드별 매장 수')
plt.xlabel('시·도')
plt.ylabel('매장 수')
plt.legend(title='브랜드')
plt.tight_layout()
plt.show()

- 경기도(1,114개, 24.8%)와 서울특별시(605개, 13.5%)를 합한 **수도권 2개 시도에 전체 매장의 38.3%**가 집중되어 있다.
- 상위 5개 시도 모두에서 **처가집**의 매장 수가 가장 많거나 상위권을 유지하고 있다.
- **페리카나**는 경기도와 서울에서 상대적으로 강세를 보이지만, 경상남도와 강원도에서는 비중이 낮다.
- **굽네**는 시도 간 분포가 비교적 고른 편이다.

---
## [탐색 4] 광역시별 브랜드 분포 비교

수도권을 제외한 5대 광역시(부산·대구·울산·광주·대전)에서 브랜드별 매장 비율이 어떻게 다른지 확인한다.

광역시는 인구 규모가 크고 주요 상권이 집중되어 있어, 브랜드 전략이 뚜렷하게 나타날 것으로 예상된다.

In [ ]:
# 5대 광역시 데이터 필터링
광역시_order = ['부산광역시', '대구광역시', '울산광역시', '광주광역시', '대전광역시']
df_metro_cities = df[df['province'].isin(광역시_order)].copy()

print('광역시별 전체 매장 수')
print(df_metro_cities['province'].value_counts())

In [ ]:
# 광역시별 브랜드 매장 수 countplot
plt.figure(figsize=(11, 5))
sns.countplot(data=df_metro_cities, x='province', hue='브랜드',
              order=광역시_order, hue_order=brand_order, palette='tab10')
plt.title('5대 광역시별 브랜드 매장 수')
plt.xlabel('광역시')
plt.ylabel('매장 수')
plt.legend(title='브랜드')
plt.tight_layout()
plt.show()

In [ ]:
# 광역시 × 브랜드 교차표
ct4 = pd.crosstab(df_metro_cities['province'], df_metro_cities['브랜드'])
ct4 = ct4.loc[광역시_order]
ct4

In [ ]:
# 광역시별 브랜드 점유율
ct4_ratio = ct4.div(ct4.sum(axis=1), axis=0) * 100
ct4_ratio.round(1)

In [ ]:
# 광역시별 브랜드 점유율 히트맵
plt.figure(figsize=(8, 4))
sns.heatmap(
    ct4_ratio.round(1),
    annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, cbar_kws={'label': '점유율 (%)'}
)
plt.title('5대 광역시별 브랜드 점유율 (%)')
plt.xlabel('브랜드')
plt.ylabel('광역시')
plt.tight_layout()
plt.show()

In [ ]:
# 광역시별 브랜드 점유율 가로 막대그래프
fig, axes = plt.subplots(1, len(광역시_order), figsize=(15, 4), sharey=True)
colors = sns.color_palette('tab10', 4)

for i, province in enumerate(광역시_order):
    vals = ct4_ratio.loc[province, brand_order]
    axes[i].barh(brand_order, vals, color=colors)
    axes[i].set_title(province.replace('광역시', '\n광역시'), fontsize=10)
    axes[i].set_xlim(0, 50)
    for j, v in enumerate(vals):
        axes[i].text(v + 0.5, j, f'{v:.0f}%', va='center', fontsize=9)

axes[0].set_ylabel('브랜드')
fig.suptitle('5대 광역시별 브랜드 점유율', y=1.02)
plt.tight_layout()
plt.show()

교차표 비율을 정리하면 다음과 같다.

| 광역시 | 굽네 | 네네 | 처가집 | 페리카나 | 1위 브랜드 |
|---|---|---|---|---|---|
| 부산 | 30.4% | 23.8% | **38.5%** | 7.3% | 처가집 |
| 대구 | 28.4% | 24.0% | 27.3% | 20.2% | 굽네 |
| 울산 | 23.9% | 23.0% | **36.3%** | 16.8% | 처가집 |
| 광주 | **37.5%** | 30.2% | 19.8% | 12.5% | 굽네 |
| 대전 | 21.0% | 17.2% | 30.6% | **31.2%** | 페리카나 |

- **부산·울산**에서는 처가집이 압도적 1위, **페리카나는 부산에서 7.3%**로 매우 낮은 비중을 보인다.
- **광주**에서는 굽네가 37.5%로 가장 강세를 보인다.
- **대전**은 처가집(30.6%)과 페리카나(31.2%)가 1·2위를 나누며 경합 중이다.
- **대구**는 4개 브랜드가 비교적 고르게 분포되어 있다.

---
## [종합] 치킨 프랜차이즈 데이터 탐색 결과

### 분석 요약

국내 치킨 프랜차이즈 4개 브랜드의 전국 매장 분포를 탐색한 결과를 정리한다.

In [ ]:
# 브랜드별 주요 지표 종합 테이블
summary = pd.DataFrame({
    '전체 매장 수': ct1.sum(axis=1),
    '수도권 매장': ct1['수도권'],
    '수도권 비율(%)': ct1_ratio['수도권'],
    '비수도권 비율(%)': ct1_ratio['비수도권'],
    '기타지방 비율(%)': ct2_ratio['기타 지방'],
})
summary = summary.loc[brand_order]
summary.round(1)

### 핵심 인사이트

**1. 전국 매장 점유율이 높은 브랜드 순서**

처가집(27.8%) > 굽네(24.7%) > 페리카나(23.9%) > 네네(23.5%) — 비교적 균등한 경쟁 구도

**2. 수도권 집중도 높은 브랜드 순서**

페리카나(48.7%) > 굽네(45.0%) > 네네(42.4%) > 처가집(37.5%)

**3. 지방 중소도시 집중도 높은 브랜드 순서**

처가집(41.5%) > 네네(39.5%) > 페리카나(38.5%) > 굽네(34.2%)

**4. 광역시별 1위 브랜드**

| 도시 | 1위 브랜드 | 특징 |
|---|---|---|
| 부산·울산 | 처가집 | 페리카나 점유율 매우 낮음 |
| 광주 | 굽네 | 굽네 전국 점유율 대비 강세 |
| 대전 | 페리카나 | 처가집과 박빙 경쟁 |
| 대구 | 굽네 | 4개 브랜드 고른 분포 |